# 04 – Entalpía de Formación a partir de los Polinomios de la NASA

La **entalpía estándar de formación** $\Delta_f H^\circ$ es el cambio de
entalpía cuando un mol de una especie se forma a partir de sus elementos en sus
estados de referencia a 298,15 K y 1 bar. Por definición, es **cero** para un
elemento en su estado de referencia (O₂, N₂, H₂, grafito, ...).

`pyglenn` ofrece un método `calculate_formation_enthalpy` — pero, como veremos,
la columna correspondiente en la base de datos está vacía. La buena noticia:
como los polinomios de la NASA ajustan la entalpía *estandarizada*, podemos
recuperar $\Delta_f H^\circ$ directamente como

$$\Delta_f H^\circ(\text{especie}) = H^\circ(298{,}15\,\mathrm{K}),$$

es decir, simplemente `calculate_properties(id, 298.15)["h_relative"]`.

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Devuelve el id en la base de datos de la especie cuyo *nombre* coincide exactamente.

    ``get_available_species`` busca subcadenas tanto en el nombre como en la
    fórmula y limita el resultado a 20 filas, por lo que nombres cortos como
    ``"O2"`` pueden quedar desplazados por entradas como ``"AL2O2"`` o
    ``"CO2"``. Para ser robustos, construimos un índice completo nombre -> id
    una sola vez (almacenado en caché durante la sesión) y buscamos el nombre
    exacto en él.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Especie {name!r} no encontrada en la base de datos")
    return _INDEX[name]

print("Constante universal de los gases R =", R, "J/(mol.K)")


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## 1. El método dedicado devuelve `None`

En la base de datos empaquetada, la columna `heat_of_formation_298K` nunca fue
rellenada, por lo que `calculate_formation_enthalpy` devuelve `None` para todas
las especies. Está implementado y listo por si una futura base de datos rellena
esa columna.

In [ ]:
with ThermochemicalCalculator() as calc:
    for name in ["H2O", "CO2", "CH4"]:
        sid = species_id(calc, name)
        print(f"{name:5s} calculate_formation_enthalpy -> "
              f"{calc.calculate_formation_enthalpy(sid)}")

## 2. Recuperando $\Delta_f H^\circ$ del polinomio

La entalpía estandarizada ya contiene la entalpía de formación, por lo que a
298,15 K *es* $\Delta_f H^\circ$. Los elementos en el estado de referencia
resultan esencialmente cero (dentro del redondeo del ajuste).

In [ ]:
def formation_enthalpy(calc, name, T=298.15):
    """Delta_f H en T (por defecto 298,15 K), en J/mol, a partir del H estandarizado."""
    return calc.calculate_properties(species_id(calc, name), T)["h_relative"]

with ThermochemicalCalculator() as calc:
    print("Elementos en estado de referencia (deben ser ~0):")
    for el in ["O2", "N2", "H2", "C(gr)"]:
        print(f"  {el:6s} {formation_enthalpy(calc, el)/1000: .4f} kJ/mol")

## 3. Una tabla validada de entalpías de formación

Calculamos $\Delta_f H^\circ$ para un conjunto de especies comunes y comparamos
con valores aceptados de la literatura (CODATA / JANAF, fase gaseosa). La
concordancia está dentro de 1 kJ/mol — los polinomios reproducen los datos de
referencia a los que fueron ajustados.

In [ ]:
# Entalpías estándar de formación de la literatura a 298,15 K, kJ/mol (fase gaseosa)
REFERENCE = {
    "H2": 0.0, "O2": 0.0, "N2": 0.0, "C(gr)": 0.0,
    "H2O": -241.83, "CO2": -393.52, "CO": -110.53, "CH4": -74.87,
    "NH3": -45.90, "NO": 91.29, "NO2": 33.10, "C2H5OH": -235.0,
}

records = []
with ThermochemicalCalculator() as calc:
    for name, ref in REFERENCE.items():
        sid = species_id(calc, name)
        M = calc.db.get_species_data(sid)["molecular_weight"]
        dhf = formation_enthalpy(calc, name) / 1000.0
        records.append({
            "especie": name, "M [g/mol]": M,
            "pyglenn [kJ/mol]": dhf, "referencia [kJ/mol]": ref,
            "error abs. [kJ/mol]": abs(dhf - ref),
        })

hf_df = pd.DataFrame(records).set_index("especie")
print(hf_df.to_string())
print(f"\nMayor error absoluto: {hf_df['error abs. [kJ/mol]'].max():.3f} kJ/mol")

## 4. Visualizando entalpías de formación

Un gráfico de barras horizontales hace evidente la división entre especies
exotérmicas ($\Delta_f H^\circ$ negativo, estables) y endotérmicas (positivo,
energéticas). Los elementos se sitúan exactamente sobre la línea del cero.

In [ ]:
order = hf_df.sort_values("pyglenn [kJ/mol]")
vals = order["pyglenn [kJ/mol]"]
colors = ["#c0392b" if v > 0 else ("#7f8c8d" if abs(v) < 1e-3 else "#2471a3")
          for v in vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(order.index, vals, color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(r"$\Delta_f H^\circ$ a 298,15 K [kJ/mol]")
ax.set_title("Entalpías estándar de formación (vía pyglenn)")
plt.show()

## 5. Solo 298,15 K es "formación"

$\Delta_f H^\circ$ se define a 298,15 K. La entalpía estandarizada $H^\circ(T)$
sigue variando con la temperatura, pero solo su valor a 298,15 K iguala la
entalpía de formación tabulada. Para el CO₂, trazamos $H^\circ(T)$ y destacamos
ese punto de referencia; el desplazamiento vertical por encima es la entalpía
*sensible* $H^\circ(T)-H^\circ(298{,}15)$ del cuaderno 03.

In [ ]:
Tgrid = np.linspace(300, 2500, 60)
with ThermochemicalCalculator() as calc:
    co2 = species_id(calc, "CO2")
    H = np.array([calc.calculate_properties(co2, t)["h_relative"] for t in Tgrid]) / 1000.0
    hf298 = formation_enthalpy(calc, "CO2") / 1000.0

fig, ax = plt.subplots()
ax.plot(Tgrid, H, label=r"$H^\circ(T)$ del CO$_2$")
ax.scatter([298.15], [hf298], color="red", zorder=5,
           label=fr"$\Delta_f H^\circ$ = {hf298:.1f} kJ/mol a 298,15 K")
ax.set_xlabel("Temperatura [K]")
ax.set_ylabel("Entalpía estandarizada [kJ/mol]")
ax.set_title("La entalpía de formación es el valor a 298,15 K de la entalpía estandarizada")
ax.legend()
plt.show()

## Resumen

- `calculate_formation_enthalpy` devuelve `None` aquí (columna de la base vacía).
- Use `calculate_properties(id, 298.15)["h_relative"]` en su lugar — equivale a
  $\Delta_f H^\circ$ y reproduce valores de la literatura con error < 1 kJ/mol.
- Como $H^\circ$ ya contiene la entalpía de formación, las entalpías de reacción
  se convierten en sumas triviales.

**A continuación:** el cuaderno 05 usa exactamente esto para calcular entalpías
de reacción y calores de combustión.